# MediScanX: End-to-End CXR Diagnostic Pipeline
**Project:** AI-Powered Multimodal Medical Scanner.<br>
**Objective:** Train the CIHMLC DenseNet121 model on the CheXpert dataset.<br> 
**Infrastructure:** This notebook serves as the cloud-GPU execution environment.

---

## Section 1: Training Configuration
This section defines the centralized configuration for the training pipeline. The `CXRTrainingConfig` dataclass ensures all hyperparameters, file paths, and model settings are strictly typed and easily accessible throughout the execution flow, eliminating magic numbers and loose variables.

In [ ]:
import os
import torch
import cv2
from dataclasses import dataclass, field

# Optimize OpenCV for multi-threading in PyTorch DataLoader
cv2_threads = 0
os.environ["OMP_NUM_THREADS"] = "1"

@dataclass
class CXRTrainingConfig:
    """
    Centralized configuration for the Chest X-Ray CIHMLC training pipeline.
    """
    # Project Info
    project_name: str = 'MediScanX-CXR-Init'
    run_name: str = 'DenseNet121-CIHMLC-70-15-15-T4'

    # Data Paths
    kaggle_dataset_root: str = "/kaggle/input/datasets/ashery/chexpert"
    kaggle_data_root: str = "/tmp/chexpert"
    csv_train_path: str = f"{kaggle_data_root}/train.csv"
    csv_valid_path: str = f"{kaggle_data_root}/valid.csv"
    
    # DataLoader Configs
    batch_size: int = 128
    num_workers: int = 4
    image_size: tuple[int, int] = (320, 320)
    
    # Model Architecture
    backbone: str = 'DenseNet121'
    num_classes: int = 14
    pretrained: bool = True
    
    # Training Hyperparameters
    epochs: int = 40
    learning_rate: float = 1e-4
    patience: int = 5
    penalty_weight: float = 1.5
    log_steps: int = 500
    weight_decay: float = 1e-4
    
    # Data Split Parameters
    train_size: float = 0.7
    val_size: float = 0.15
    test_size: float = 0.15
    random_seed: int = 42
    
    

    # Standard CheXpert labels in exact categorical order
    CHEXPERT_LABELS: list[str] = field(default_factory=lambda: [
        "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
        "Lung Lesion", "Edema", "Consolidation", "Pneumonia", "Atelectasis", 
        "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"
    ])

    # Clinical Taxonomy (Parent Index -> Child Index)
    # Enforces multi-level hierarchical rules based on the CheXpert schema:
    # 1: 'Enlarged Cardiomediastinum' -> 2: 'Cardiomegaly'
    # 3: 'Lung Opacity' -> 4: 'Lung Lesion', 5: 'Edema', 6: 'Consolidation', 8: 'Atelectasis'
    # 6: 'Consolidation' -> 7: 'Pneumonia'
    HIERARCHY_PAIRS: list[tuple[int, int]] = field(default_factory=lambda: [
        (1, 2),
        (3, 4),
        (3, 5),
        (3, 6),
        (3, 8),
        (6, 7)
    ])
    
    # Hardware
    device: torch.device = field(
        default_factory=lambda: torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    )

# Initialize Global Configuration
cfg = CXRTrainingConfig()
print(f"Executing MediScan CXR Pipeline on: {cfg.device}")
print("Centralized configuration for the training pipeline defined.")

## Section 1.1: High-Speed Data Caching (NVMe Transfer)
Cloud environments often suffer from network-attached storage bottlenecks. This module safely bypasses standard OS-level directory scanning by using the clinical CSV annotations as a direct map, blasting the dataset to the local `/tmp/` NVMe SSD using highly concurrent thread pools.

In [ ]:
import os
import shutil
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# Config constants
SOURCE_ROOT = cfg.kaggle_dataset_root
DEST_ROOT = cfg.kaggle_data_root
NUM_WORKERS = 64

def csv_guided_nvme_transfer(
    source_root: str = SOURCE_ROOT,
    dest_root: str = DEST_ROOT,
    num_workers: int = NUM_WORKERS
) -> None:
    """
    Transfer CheXpert dataset from Kaggle Servers to local Kaggle Notebook NVMe storages
    using CSV guides.
    
        Reads the train.csv and valid.csv files to get a list of image paths, then uses
    parallel threads to copy all the images and CSV files to the destination directory.

    Args:
        source_root (str): Path to the Kaggle dataset directory. Defaults to SOURCE_ROOT.
        dest_root (str): Destination path on local NVMe storage. Defaults to DEST_ROOT.
        num_workers (int): Number of parallel threads for copying files. Defaults to NUM_WORKERS.

    Returns:
        None
    """
    if os.path.exists(dest_root):
        print(f"Dataset already exists at {dest_root}. Skipping transfer.")
        return
    
    print("Bypassing network scan by reading CSV map...")
    
    # Read the maps
    train_df = pd.read_csv(f"{source_root}/train.csv")
    # Load valid.csv if it exists in that dataset structure
    valid_csv_path = f"{source_root}/valid.csv"
    if os.path.exists(valid_csv_path):
        valid_df = pd.read_csv(valid_csv_path)
        all_paths = pd.concat([train_df['Path'], valid_df['Path']]).values
    else:
        all_paths = train_df['Path'].values
    
    copy_tasks = []
    
    # Build exact file paths in memory
    for path in all_paths:
        cleaned_path = path.replace("CheXpert-v1.0-small/", "")
        src = os.path.join(source_root, cleaned_path)
        dst = os.path.join(dest_root, cleaned_path)
        copy_tasks.append((src, dst))
        
    total_files = len(copy_tasks)
    print(f"Mapped {total_files:,} files. Copying to NVMe with 64 threads...")
    
    def copy_worker(task):
        """Copy a single file and create parent directories if needed."""
        src, dst = task
        # Create the parent directory
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        # Copy the image if it exists
        if os.path.exists(src):
            shutil.copy2(src, dst)
        return True
    
    # Spawn 64 threads to saturate Kaggle's network bandwidth
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        list(tqdm(executor.map(copy_worker, copy_tasks), total=total_files, desc="Copying to /tmp/", unit="file"))
        
    # Finally, copy the CSV files themselves over to the SSD
    shutil.copy2(f"{source_root}/train.csv", f"{dest_root}/train.csv")
    if os.path.exists(valid_csv_path):
        shutil.copy2(valid_csv_path, f"{dest_root}/valid.csv")
        
    print("Transfer complete! Data is now cached on the local SSD for faster I/O.")

# Execute the hyper-fast transfer using our global configuration
csv_guided_nvme_transfer()       

## Section 1.2: Experiment Telemetry Module

This section defines the `ExperimentTracker`, an MLOps abstraction that handles authentication and session management for Weights & Biases.

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient
from dataclasses import asdict

class ExperimentTracker:
    """
    Manages MLOps telemetry, authentication and experiment lifecycle via Weights & Biases.
    """
    
    @staticmethod
    def initialize(cfg: CXRTrainingConfig) -> None:
        """Authenticates the environment using Kaggle Secrets and initializes WandB.

        Args:
            cfg (CXRTrainingConfig): The centralized config object to log as hyperparameters.
        """
        try:
            # Fetch the API key from the Kaggle environment
            user_secrets = UserSecretsClient()
            wandb_api = user_secrets.get_secret('wandb_api_key')
            wandb.login(key=wandb_api)
            print("Successfully authenticated with Weights and Biases!")
        except Exception as e:
            print(f"WandB Auth Error: {e}. Ensure 'wandb_api_key' is added to Kaggle Secrets.")
            
        # Initialize the tracking session
        wandb.init(
            project=cfg.project_name,
            name=cfg.run_name,
            config=asdict(cfg)
        )
        
    @staticmethod
    def close() -> None:
        """Safely terminates the active WandB session to sync final logs."""
        wandb.finish()
            
print("Experiment Telemetry Module defined successfully.")

## Section 2: Preprocessing Pipeline
This section defines the deterministic clinical preprocessing sequence applied to the chest X-rays. It standardizes the radiographic inputs through contrast enhancement (CLAHE) and tensor transformations, establishing the foundational data format required for neural network ingestion.

In [ ]:
import cv2
import numpy as np
from torchvision import transforms

class ApplyCLAHE:
    """
    A custom PyTorch transform that applies Contrast Limited Adaptive Histogram Equalization (CLAHE) to a medical image.
    
    This class standardizes radiographic inputs by normalizing illumination discrepancies and enhancing local contrast 
    within lung fields, mitigating artifacts from legacy X-ray machines.
    """
    
    def __init__(self, clip_limit: float = 2.0, tile_grid_size: tuple[int, int] = (8, 8)) -> None:
        """
        Initializes the ApplyCLAHE transform object.

        Args:
            clip_limit (float): Sets the threshold for contrast limiting.
            tile_grid_size (tuple[int, int]): Sets the grid size for localized equalization.
        """
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size
        self.clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        
    def __call__(self, img: np.ndarray) -> np.ndarray:
        """
        Executes CLAHE and stacks the output into a 3-channel array for transfer learning.

        Args:
            img (np.ndarray): The input grayscale image array.

        Returns:
            np.ndarray: The contrast-enhanced, 3-channel RGB image array.
            
        Raises:
            TypeError: If the input is not a NumPy array. 
        """
        if not isinstance(img, np.ndarray):
            raise TypeError(f"ApplyCLAHE expects a numpy.ndarray, but got {type(img)}")
        
        # Ensure the image is 8-bit grayscale as required by OpenCV CLAHE
        if img.dtype != np.uint8:
            img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
            img = img.astype(np.uint8)
            
        # Apply the CLAHE algorithm
        enhanced_img = self.clahe.apply(img)
        
        # Convert to 3-channel representation directly in OpenCV
        enhanced_img_3c = cv2.cvtColor(enhanced_img, cv2.COLOR_GRAY2RGB)
        
        return enhanced_img_3c
    
    def __repr__(self) -> str:
        """Return a string representation of the transform object."""
        return f"{self.__class__.__name__}(clip_limit={self.clip_limit}, tile_grid_size={self.tile_grid_size})"


class RadiographicPipeline:
    """
    A factory class constructing the base preprocessing pipeline for Chest X-Rays prior to model ingestion.
    Provides a unified transform sequence to ensure parity between training and inference environments.
    """
    
    @staticmethod
    def get_base_transforms(resize_dim: tuple[int, int] = (320, 320)) -> transforms.Compose:
        """
        Constructs the sequence of base transforms including CLAHE, tensor conversion, and resizing.

        Args:
            resize_dim (tuple[int, int]): Target dimensions for the neural network.

        Returns:
            transforms.Compose: The chained PyTorch transformations.
        """
        return transforms.Compose([
            ApplyCLAHE(clip_limit=2.0, tile_grid_size=(8, 8)),
            transforms.ToTensor(),
            transforms.Resize(resize_dim, antialias=True)
        ])

print("Preprocessing pipeline initialized successfully.")

## Section 3: Dataset Implementation
This section defines the custom PyTorch Dataset responsible for handling the CheXpert database. It manages image retrieval from the local drive, parses the multi-label clinical CSV annotations, and enforces the U-Ones triage policy by strictly mapping uncertain pathology labels to positive cases.

In [ ]:
import os
import torch
import cv2
import numpy as np
import pandas as pd
from typing import Optional, Callable
from torch.utils.data import Dataset

class CheXpertDataset(Dataset):
    """
    A custom PyTorch Dataset implementation for the CheXpert database.
    
    This class handles the parsing of clinical annotations in CSV format, loads the corresponding 
    high-resolution X-ray images from disk using OpenCV, applies defined preprocessing transforms, 
    and extracts the multi-label pathology vectors.

    Attributes:
        annotations (pd.DataFrame): The parsed CSV data containing paths and clinical labels.
        root_dir (str): The base directory path where the image dataset is stored.
        transform (Callable, optional): A PyTorch transform to apply to the images.
    """
    
    def __init__(self, csv_file: str, root_dir: str, transform: Optional[Callable] = None) -> None:
        """
        Initializes the CheXpert Dataset.

        Args:
            csv_file (str): Path to the train.csv or valid.csv file.
            root_dir (str): Directory containing the CheXpert image folders.
            transform (Callable, optional): Optional transform applied to a sample.
        """
        super().__init__()
        
        # Load the CSV. CheXpert labels sometimes contain -1 (uncertain).
        # U-Ones Policy: Map -1 (uncertain) to 1 (positive) for triage safety.
        self.annotations = pd.read_csv(csv_file)
        self.annotations = self.annotations.fillna(0)
        self.annotations = self.annotations.replace(-1, 1)
        
        self.root_dir = root_dir
        self.transform = transform
        
    def __len__(self) -> int:
        """Returns the total number of patient scans in the dataset."""
        return len(self.annotations)
    
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Retrieves a single medical image and its corresponding pathology labels.

        Args:
            idx (int): The index of the item to retrieve.

        Returns:
            tuple[torch.Tensor, torch.Tensor]: A tuple containing the processed image tensor 
                                                and the multi-hot label tensor.

        Raises:
            FileNotFoundError: If the resolved image path does not exist on disk.
        """
        if torch.is_tensor(idx):
            idx = idx.tolist()
        
        # The CheXpert 'Path' column contains the relative path
        original_path = self.annotations.iloc[idx]['Path']
        cleaned_path = original_path.replace('CheXpert-v1.0-small/', '')
        img_path = os.path.join(self.root_dir, cleaned_path)
        
        # Load the image strictly in grayscale
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        if image is None:
            raise FileNotFoundError(f"OpenCV could not read the image at: {img_path}")

        # Extract the 14 pathology labels (columns 5 to 18 in CheXpert)
        labels = self.annotations.iloc[idx, 5:19].values.astype(np.float32)
        
        if self.transform:
            image = self.transform(image)
            
        label_tensor = torch.tensor(labels, dtype=torch.float32)
        
        return image, label_tensor

print("CheXpert Dataset class defined successfully.")

## Section 3.1: Data Routing Module

This section defines the `CXRDataModule` factory class. Crucially, it replaces standard random splitting with a patient-aware `GroupShuffleSplit`. By extracting the Patient ID from the CheXpert file paths, it ensures that all scans from a single patient are routed exclusively into either the train, validation, or test set. This strictly prevents patient-level anatomical data leakage, ensuring the model's validation metrics reflect true clinical generalization.

In [ ]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import GroupShuffleSplit
from typing import Tuple

class CXRDataModule:
    """
    Orchestrates the instantiation and routing of datasets and dataloaders.
    
    Enforces strict patient-level grouping during the train/val/test split to prevent 
    anatomical data leakage across evaluation boundaries.
    """
    
    def __init__(self, cfg: CXRTrainingConfig) -> None:
        """
        Initializes the CXRDataModule with the global configuration.

        Args:
            cfg (CXRTrainingConfig): The centralized configuration object.
        """
        self.cfg = cfg
        self.base_transforms = RadiographicPipeline.get_base_transforms(self.cfg.image_size)
        
    def setup(self) -> Tuple[DataLoader, DataLoader, DataLoader]:
        """
        Constructs the datasets, performs the patient-aware splits, and builds the dataloaders.

        Returns:
            Tuple[DataLoader, DataLoader, DataLoader]: The train, validation, and test dataloaders.
        """
        # Instantiate the full dataset with CPU-bound deterministic transforms
        full_dataset = CheXpertDataset(
            csv_file=self.cfg.csv_train_path,
            root_dir=self.cfg.kaggle_data_root,
            transform=self.base_transforms
        )
        
        # Extract Patient IDs to act as isolation groups
        # CheXpert paths look like: 'CheXpert-v1.0-chexpert/train/patient00001/study1/view1_frontal.jpg'
        # Splitting by '/' and taking index 2 reliably extracts the 'patientXXXXX' string
        groups = full_dataset.annotations['Path'].apply(lambda x: x.split('/')[2]).values
        
        # First Split: Isolate Training data from Evaluation data (Val + Test combined)
        eval_size = self.cfg.val_size + self.cfg.test_size
        gss_train_eval = GroupShuffleSplit(n_splits=1, test_size=eval_size, random_state=self.cfg.random_seed)
        
        train_idx, eval_idx = next(gss_train_eval.split(full_dataset.annotations, groups=groups))
        
        # Second Split: Divide the Evaluation pool into Validation and Test sets
        test_ratio = self.cfg.test_size / eval_size
        eval_groups = groups[eval_idx]
        gss_val_test = GroupShuffleSplit(n_splits=1, test_size=test_ratio, random_state=self.cfg.random_seed)
        
        val_idx_relative, test_idx_relative = next(gss_val_test.split(eval_idx, groups=eval_groups))
        
        # Map relative evaluation indices back to the absolute original dataset indices
        val_idx = eval_idx[val_idx_relative]
        test_idx = eval_idx[test_idx_relative]
        
        # Construct PyTorch Subsets
        train_dataset = Subset(full_dataset, train_idx)
        val_dataset = Subset(full_dataset, val_idx)
        test_dataset = Subset(full_dataset, test_idx)
        
        # Build High-Performance DataLoaders
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.cfg.batch_size,
            shuffle=True, # Shuffles batches, but patients remain strictly in the training set
            num_workers=self.cfg.num_workers,
            pin_memory=True if self.cfg.device.type == 'cuda' else False,
            drop_last=True
        )
        
        val_loader = DataLoader(
            val_dataset,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            num_workers=self.cfg.num_workers,
            pin_memory=True if self.cfg.device.type == 'cuda' else False,
            drop_last=False
        )
        
        test_loader = DataLoader(
            test_dataset,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            num_workers=self.cfg.num_workers,
            pin_memory=True if self.cfg.device.type == 'cuda' else False,
            drop_last=False
        )
        
        print(f"Split complete. Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
        return train_loader, val_loader, test_loader

print("CXRDataModule defined successfully.")

## Section 4: Model Architecture (DenseNet121_CIHMLC)

This section defines the core computer vision architecture. It instantiates a DenseNet121 backbone, strips its default classifier, and appends a custom `Conv2d` head (512 filters) for finer structural feature extraction. The `forward` pass is explicitly structured to yield both the classification logits and the spatial feature maps, fulfilling the architectural requirements for downstream Grad-CAM++ explainability.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class DenseNet121_CIHMLC(nn.Module):
    """
    Clinically-Inspired Hierarchical Multi-Label Classification Model.
    
    Utilizes a DenseNet121 backbone optimized for feature reuse. The architecture features
    a custom convolutional head and a dual-output forward pass to enable offline 
    Explainable AI (Grad-CAM++) on mobile edge devices.
    
    Attributes:
        features (nn.Sequential): The foundational DenseNet121 feature extractor blocks.
        custom_conv (nn.Conv2d): Dimensionality reduction and fine structural extraction layer.
        bn (nn.BatchNorm2d): Batch normalization for the custom convolutional head.
        relu (nn.ReLU): Non-linear activation for the custom conv layer.
        global_avg_pool (nn.AdaptiveAvgPool2d): Condenses spatial dimensions to (1, 1).
        dropout (nn.Dropout): Regularization layer.
        classifier (nn.Linear): The final dense layer mapping to the clinical pathologies.
    """
    def __init__(self, num_classes: int=14, pretrained: bool=True):
        """
        Initializes the DenseNet121_CIHMLC architecture.

        Args:
            num_classes (int, optional): The number of output diagnostic labels. Defaults to 14.
            pretrained (bool, optional): Whether to initialize the ImageNet weights. Defaults to True.
        """
        super(DenseNet121_CIHMLC, self).__init__()
        
        # Load the foundation DenseNet121 backbone
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        densenet = models.densenet121(weights=weights)
        
        # Extract the feature blocks
        self.features = densenet.features
        
        # Append Custom Conv2D Layer
        # DenseNet121's feature extractor naturally outputs 1024 channels.
        # We compress this to 512 filters to extract finer structural details and reduce parameters
        self.custom_conv = nn.Conv2d(
            in_channels=1024,
            out_channels=512,
            kernel_size=3,
            padding=1,
            bias=False
        )
        # BatchNorm and ReLU for stabilization and non-linearity
        self.bn = nn.BatchNorm2d(512)
        self.relu = nn.ReLU(inplace=True)
        
        # Global Average Pooling: Condenses the spatial dimensions (H,W) to (1,1) while preserving the 512 feature maps
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Add dropout for regularization
        self.dropout = nn.Dropout(p=0.4)
        
        # Final Classification layer
        self.classifier = nn.Linear(in_features=512, out_features=num_classes)
        
    def forward(self, x: torch.Tensor) -> tuple:
        """
        Executes the dual-output forward pass.

        Args:
            x (torch.Tensor): A batch of 3-channel X-ray images. Shape: [B, 3, H, W]

        Returns:
            tuple: 
                - logits (torch.Tensor): The raw, unactivated classification scores. Shape: [B, 14]
                - spatial_features (torch.Tensor): The raw spatial maps for Grad-CAM++. Shape: [B, 512, H', W']
        """
        # Pass through the deep DenseNet blocks
        x = self.features(x)
        
        # Pass through the custom structural extraction head
        spatial_features = self.relu(self.bn(self.custom_conv(x)))
        
        # Pool, flatten and classify
        pooled = self.global_avg_pool(spatial_features)
        flattened = torch.flatten(pooled, 1)
        
        # Apply the dropout before the final layer
        flattened = self.dropout(flattened)
        logits = self.classifier(flattened)
        
        # Return both artifacts for the inference engine
        return logits, spatial_features

print("DenseNet121 CIHMLC Model Architecture defined.")

## Section 5: Clinical Imbalance Weighting

This section handles the statistical mechanics. It defines the `ClassWeightCalculator` utility, which dynamically computes class-specific positive weights to mathematically counteract the severe class imbalances inherent in the clinical dataset.

In [ ]:
import torch
import numpy as np
import pandas as pd

class ClassWeightCalculator:
    """
    Utility class for calculating positive loss weights in imbalanced clinical datasets.
    """
    
    @staticmethod
    def compute_pos_weights(df: pd.DataFrame, num_classes: int = 14) -> torch.Tensor:
        """
        Calculates the ratio of negative to positive samples for each diagnostic class.
        
        This creates a weight vector optimized for `BCEWithLogitsLoss(pos_weight=...)` to scale 
        the gradient of minority positive classes during back propagation.
        
        Args:
            df (pd.DataFrame): The raw training dataframe containing clinical annotations.
            num_classes (int): Total number of pathology labels to evaluate. Defaults to 14.
            
        Returns:
            torch.Tensor: A 1D tensor of strictly typed positive weights.
        """
        # Clean the dataframe dynamically: fill NaNs with 0, apply U-Ones policy (-1 to 1)
        df_clean = df.fillna(0).replace(-1, 1)
        
        # Extract the label matrix (CheXpert pathology labels strictly start at column index 5)
        labels = df_clean.iloc[:, 5:5 + num_classes].values
        
        # Calculate class distributions
        pos_counts = np.sum(labels == 1, axis=0)
        neg_counts = np.sum(labels == 0, axis=0)
        
        # Add a small epsilon to the denominator to prevent division by zero
        pos_weights = neg_counts / (pos_counts + 1e-7)
        
        return torch.tensor(pos_weights, dtype=torch.float32)

print("Clinical Taxonomies and Weight Calculator defined successfully.")

## Section 6: The Hierarchical Loss Function (HBCE)

This section defines the `HBCELoss`, which combines standard weighted Binary Cross-Entropy (BCE) with a custom hierarchical penalty mechanism. It iterates through the defined `HIERARCHY_PAIRS`. If the neural network assigns a higher probability to a child diagnostic class than its mandatory parent class, a mathematical penalty applies. This enforces anatomically correct associations and logical constraints during model optimization.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class HBCELoss(nn.Module):
    """
    Hierarchical Binary Cross-Entropy Loss.
    
    Attributes:
        pos_weight (torch.Tensor): Weights to counter class imbalance.
        hierarchy_pairs (list): List of (parent_idx, child_idx) tuples.
        penalty_weight (float): Multiplier for the hierarchical violation penalty.
    """
    
    def __init__(self, pos_weight: torch.Tensor, hierarchy_pairs: list, penalty_weight: float = 1.0):
        super(HBCELoss, self).__init__()
        self.pos_weight = pos_weight
        self.hierarchy_pairs = hierarchy_pairs
        self.penalty_weight = penalty_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Calculates the penalized loss using the cascading taxonomy map.
        """
        # Label Smoothing: 1.0 -> 0.95, 0.00 -> 0.05
        smoothed_targets = targets * 0.90 + 0.05
        
        # Standard Weighted Binary Cross Entropy with label-smoothed targets
        bce_loss = F.binary_cross_entropy_with_logits(
            logits, smoothed_targets, pos_weight=self.pos_weight, reduction='mean'
        )
        
        # Compute probabilities for the hierarchy check
        probs = torch.sigmoid(logits)
        
        # Calculate Hierarchical Penalties using the provided tuples
        penalty = torch.tensor(0.0, device=logits.device)
        
        for parent_idx, child_idx in self.hierarchy_pairs:
            # Violation occurs if P(Child) > P(Parent)
            # ReLU zeros out the tensor if P(Parent) is correctly higher than P(Child)
            violation = F.relu(probs[:, child_idx] - probs[:, parent_idx])
            penalty += torch.mean(violation)
            
        # Total Loss computation
        total_loss = bce_loss + (self.penalty_weight * penalty)
        return total_loss

print("Defined the custom Hierarchical Loss function based on Hierarchical Binary Cross-Entropy.")

## Section 7: The Model Trainer

This section defines the execution engine responsible for model orchestration. It manages the forward and backward passes, applies GPU-accelerated batch augmentations, computes complex multi-label clinical metrics (AUC, F1, Precision, Recall) across all data splits, and enforces Early Stopping with real-time Weights & Biases (WandB) synchronization.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import wandb
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader
from torch.optim import Optimizer
from typing import Any
import torchvision.transforms.v2 as v2

class CIHMLCTrainer:
    """
    Trainer engine managing model optimization, evaluation metrics, and 
    experiment telemetry tracking via Weights & Biases.
    """
    def __init__(
        self, 
        model: nn.Module, 
        train_loader: DataLoader, 
        val_loader: DataLoader, 
        test_loader: DataLoader, 
        criterion: nn.Module, 
        optimizer: Optimizer, 
        scheduler: Any, 
        config: CXRTrainingConfig
    ) -> None:
        """
        Initializes the trainer engine with the required PyTorch components and data splits.

        Args:
            model (nn.Module): The PyTorch neural network to be trained.
            train_loader (DataLoader): Iterable over the training subset.
            val_loader (DataLoader): Iterable over the validation subset.
            test_loader (DataLoader): Iterable over the held-out test subset.
            criterion (nn.Module): The hierarchical loss function (HBCELoss).
            optimizer (Optimizer): The optimization algorithm.
            scheduler (Any): Learning rate decay scheduler.
            config (CXRTrainingConfig): The centralized configuration object.
        """
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.cfg = config
        
        self.best_auc = 0.0
        self.global_step = 0
        device_type = getattr(self.cfg.device, "type", "cuda")
        self.scaler = torch.amp.GradScaler(enabled=(device_type == "cuda"))
        
        # Batched GPU Augmentations (Hardware Accelerated)
        self.gpu_train_aug = v2.Compose([
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomRotation(degrees=10),
            v2.ColorJitter(brightness=0.2, contrast=0.2),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        # Validation Normalization matching Inference parameters
        self.gpu_val_aug = v2.Compose([
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def _calculate_metrics(self, targets: np.ndarray, probs: np.ndarray, preds: np.ndarray) -> tuple[float, float, float, float]:
        """
        Internal helper to calculate macro-averaged classification metrics.

        Args:
            targets (np.ndarray): Ground truth binary labels.
            probs (np.ndarray): Continuous probability predictions [0, 1].
            preds (np.ndarray): Thresholded binary predictions {0, 1}.

        Returns:
            tuple[float, float, float, float]: Mean AUC, F1 Score, Precision, and Recall.
        """
        try: 
            mean_auc = float(roc_auc_score(targets, probs, average='macro'))
            f1 = float(f1_score(targets, preds, average='macro', zero_division=0))
            precision = float(precision_score(targets, preds, average='macro', zero_division=0))
            recall = float(recall_score(targets, preds, average='macro', zero_division=0))
        except ValueError:
            mean_auc, f1, precision, recall = 0.0, 0.0, 0.0, 0.0
        
        return mean_auc, f1, precision, recall
    
    def train_epoch(self) -> tuple[float, float, float, float ,float]:
        """
        Executes one complete forward and backward pass over the training dataset.

        Returns:
            tuple[float, float, float, float, float]: Epoch Loss, AUC, F1, Precision, Recall.
        """
        self.model.train()
        running_loss = 0.0
        step_running_loss = 0.0
        all_targets, all_probs, all_preds = [], [], []

        total_steps = len(self.train_loader)
        log_step_interval = 50
        console_step_interval = 400 
        
        loop = tqdm(self.train_loader, desc='Training', leave=False)
        for i, (images, labels) in enumerate(loop):
            # Non-blocking transfer to GPU
            images = images.to(self.cfg.device, non_blocking=True)
            labels = labels.to(self.cfg.device, non_blocking=True)
            
            # Apply hardware-accelerated augmentations
            images = self.gpu_train_aug(images)
            
            self.optimizer.zero_grad()
        
            # Automatic Mixed Precision Context Manager
            with torch.amp.autocast('cuda'):
                logits, _ = self.model(images)
                loss = self.criterion(logits, labels)

            # Scale gradients, backward pass, and optimizer step using AMP
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            current_loss = loss.item()
            running_loss += current_loss
            step_running_loss += current_loss

            # Granular Step Logging to WandB
            if (i + 1) % log_step_interval == 0:
                avg_step_loss = step_running_loss / log_step_interval
                wandb.log({
                    "train/step_loss": avg_step_loss,
                    "global_step": self.global_step
                })
                
                # Clean console print based on the larger interval
                if (i + 1) % console_step_interval == 0:
                    tqdm.write(f"  -> Step [{i+1}/{total_steps}] | Moving Avg Loss: {avg_step_loss:.4f}")
                    
                step_running_loss = 0.0

            self.global_step += 1

            # Store predictions for Epoch-level metric calculation
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs > 0.5).float()
                
                all_targets.append(labels.cpu().numpy())
                all_probs.append(probs.cpu().numpy())
                all_preds.append(preds.cpu().numpy())
                
            loop.set_postfix(loss=current_loss)
            
        # Compile epoch-level arrays
        all_targets = np.vstack(all_targets)
        all_probs = np.vstack(all_probs)
        all_preds = np.vstack(all_preds)
            
        epoch_loss = running_loss / total_steps
        auc, f1, precision, recall = self._calculate_metrics(all_targets, all_probs, all_preds)

        return epoch_loss, auc, f1, precision, recall
        
    def evaluate(self, dataloader: DataLoader, desc: str = 'Validating') -> tuple[float, float, float, float, float]:
        """
        Evaluates generalized model performance on a specified subset.

        Args:
            dataloader (DataLoader): The validation or test dataloader.
            desc (str): Description string for the progress bar. Defaults to 'Validating'.

        Returns:
            tuple[float, float, float, float, float]: Epoch Loss, AUC, F1, Precision, Recall.
        """
        self.model.eval()
        total_loss = 0.0
        all_targets, all_probs, all_preds = [], [], []
        
        with torch.no_grad():
            for images, labels in tqdm(dataloader, desc=desc, leave=False):
                images = images.to(self.cfg.device, non_blocking=True)
                labels = labels.to(self.cfg.device, non_blocking=True)

                images = self.gpu_val_aug(images)
                
                logits, _ = self.model(images)
                loss = self.criterion(logits, labels)
                total_loss += loss.item()
                
                probs = torch.sigmoid(logits)
                preds = (probs > 0.5).float()
                
                all_targets.append(labels.cpu().numpy())
                all_probs.append(probs.cpu().numpy())
                all_preds.append(preds.cpu().numpy())
                
        all_targets = np.vstack(all_targets)
        all_probs = np.vstack(all_probs)
        all_preds = np.vstack(all_preds)
            
        epoch_loss = total_loss / len(dataloader)
        auc, f1, precision, recall = self._calculate_metrics(all_targets, all_probs, all_preds)
        
        return epoch_loss, auc, f1, precision, recall
    
    def execute_training(self, save_path: str='densenet_cihmlc.pth') -> None:
        """
        Orchestrates the training loop, early stopping, and WandB telemetry logging.

        Args:
            save_path (str): The local disk path to save the best model weights.
        """
        patience_counter = 0
        
        for epoch in range(self.cfg.epochs):
            print(f"\n========== Epoch {epoch+1}/{self.cfg.epochs} ===========")
            
            t_loss, t_auc, t_f1, t_precision, t_recall = self.train_epoch()
            v_loss, v_auc, v_f1, v_precision, v_recall = self.evaluate(self.val_loader, desc='Validating')

            self.scheduler.step(v_loss)
            current_lr = self.optimizer.param_groups[0]['lr']

            # Epoch-Level WandB Logging
            wandb.log({
                "epoch": epoch + 1,
                "learning_rate": current_lr,
                "train/loss": t_loss, "train/auc": t_auc, "train/f1": t_f1, "train/precision": t_precision, "train/recall": t_recall,
                "val/loss": v_loss, "val/auc": v_auc, "val/f1": v_f1, "val/precision": v_precision, "val/recall": v_recall,
            })
            
            print(f"TRAIN -> Loss: {t_loss:.4f} | AUC: {t_auc:.4f} | F1: {t_f1:.4f} | Precision: {t_precision:.4f} | Recall: {t_recall:.4f}")
            print(f"VALID -> Loss: {v_loss:.4f} | AUC: {v_auc:.4f} | F1: {v_f1:.4f} | Precision: {v_precision:.4f} | Recall: {v_recall:.4f}")
            
            if v_auc > self.best_auc:
                self.best_auc = v_auc
                torch.save(self.model.state_dict(), save_path)
                print(f"*** New Best Model Saved (Val AUC: {v_auc:.4f}) ***")
                patience_counter = 0
            else:
                patience_counter += 1
                
            if patience_counter >= self.cfg.patience:
                print(f"Early stopping triggered after {self.cfg.patience} epochs without improvement.")
                break
            
        print("\n========== Commencing Final Test Set Evaluation ===========")
        self.model.load_state_dict(torch.load(save_path))
        test_loss, test_auc, test_f1, test_precision, test_recall = self.evaluate(self.test_loader, desc='Testing')
        
        wandb.log({
            "test/loss": test_loss, "test/auc": test_auc, "test/f1": test_f1,
            "test/precision": test_precision, "test/recall": test_recall 
        })
        
        print(f"TEST -> Loss: {test_loss:.4f} | AUC: {test_auc:.4f} | F1: {test_f1:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f}")

print("CIHMLC Trainer engine defined successfully.")

## Section 8: Pipeline Execution Engine

This is the main entry point for the training pipeline. It cleanly instantiates the configuration, initializes the telemetry tracker, routes the data via the `CXRDataModule`, constructs the model architecture, and triggers the `CIHMLCTrainer`. By relying strictly on our OOP abstractions, this block remains highly readable and maintainable.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch._dynamo
torch._dynamo.config.suppress_errors = True

def main():
    """
    Executes the end-to-end MediScanX Chest X-Ray training pipeline.
    """
    # Initialize MLOps Telemetry
    ExperimentTracker.initialize(cfg)
    
    # Setup Data Routing (Patient-Aware Data Splits and DataLoaders)
    print("Initializing Data Module and performing patient-aware splits...")
    data_module = CXRDataModule(cfg)
    train_loader, val_loader, test_loader = data_module.setup()
    
    # Instantiate Model Architecture
    model = DenseNet121_CIHMLC(num_classes=cfg.num_classes, pretrained=cfg.pretrained)
    
    # Enable multi-GPU support if available
    if torch.cuda.device_count() > 1:
        print(f"Accelerating training across {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
        
    model = model.to(cfg.device)
    
    # Compile the model for native C++ kernel fusion
    print("Compiling model via torch.compile()...")
    model = torch.compile(model)
    
    # Initialize Dynamic Loss Function
    # Extract the actual training dataframe from the PyTorch subset to calculate class imbalance
    train_indices = train_loader.dataset.indices
    df_train = train_loader.dataset.dataset.annotations.iloc[train_indices]
    
    pos_weight = ClassWeightCalculator.compute_pos_weights(df_train, num_classes=cfg.num_classes).to(cfg.device)
    hbce_criterion = HBCELoss(pos_weight=pos_weight, hierarchy_pairs=cfg.HIERARCHY_PAIRS, penalty_weight=cfg.penalty_weight)
    
    # Optimizer and Scheduler
    optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)
    
    # Execute Training Loop
    trainer_cfg = CIHMLCTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        criterion=hbce_criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        config=cfg
    ) 
    
    trainer.execute_training(save_path='densenet_cihmlc.pth')
    
    # Clean up Telemetry
    ExperimentTracker.close()
if __name__ == "__main__":
    main()